# 🤖 Google Gemini API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — Gemini API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **Google API 키**를 입력하고,
**프롬프트를 작성**해서 Gemini 모델의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, Gemini!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 역할 부여 |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_output_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** Google AI Studio에서 발급받으세요.
> 키는 `AIza...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 Google Generative AI 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [1]:
# Google Generative AI 라이브러리 설치
%pip install google-generativeai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import google.generativeai as genai
    print("✅ 설치 완료 · google.generativeai")
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

Note: you may need to restart the kernel to use updated packages.
✅ 설치 완료 · google.generativeai


c:\Users\LRO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\LRO\AppData\Local\Temp\ipykernel_16336\1162930208.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [2]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key를 입력하세요 (AIza...): ")

print("✅ API 키가 등록되었습니다.")

✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`model`은 우리가 Gemini에게 요청을 보내는 **대상**입니다.
위에서 등록한 환경변수(`GOOGLE_API_KEY`)를 사용해 설정합니다.

> ⚠️ **모델 선택 주의**
> - `gemini-2.0-flash` : 실습에 가장 무난하고 빠른 모델
> - `gemini-1.5-flash` : 안정적인 경량 모델
> - `gemini-1.5-pro` : 더 강력하지만 비용/속도 고려 필요


In [3]:
import google.generativeai as genai

# 환경변수에서 API 키를 읽어 설정
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# 사용할 모델 이름
MODEL = "gemini-2.0-flash"

model = genai.GenerativeModel(MODEL)

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gemini-2.0-flash


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
response = model.generate_content("질문 내용")
```

- Gemini는 `generate_content(...)`로 요청을 보냅니다.
- 답변 텍스트는 `response.text` 안에 들어 있습니다.
- 출력 길이(`max_output_tokens`) 같은 옵션은 6단계에서 다룹니다.


In [4]:
response = model.generate_content("안녕하세요! 당신을 한 문장으로 소개해 주세요.")

# 답변 텍스트 꺼내기
print(response.text)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 24.849950566s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quo

### 응답 안에는 뭐가 들어있을까?

답변 말고도 사용량 정보가 함께 옵니다.
Gemini는 응답 객체에서 `usage_metadata`를 확인할 수 있습니다.

In [ ]:
print("입력 토큰 수 :", getattr(response, "usage_metadata", None))

## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 Gemini에 역할 지시를 포함해 보낼 수 있습니다.


In [ ]:
# Gemini 호출 형식
'''
response = model.generate_content(
    "질문 내용",
    generation_config=genai.types.GenerationConfig(
        temperature=0.7,
        max_output_tokens=1024,
    ),
)
'''

In [ ]:
def ask(prompt, system=None, max_output_tokens=1024, temperature=0.7):
    """프롬프트를 보내고 Gemini의 답변 텍스트를 돌려줍니다."""
    instruction = system or "당신은 도움이 되는 AI 어시스턴트입니다."
    full_prompt = f"{instruction}\n\n사용자 질문: {prompt}" if system else prompt

    response = model.generate_content(
        full_prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        ),
    )
    return response.text

print("✅ ask() 함수 준비 완료")

👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [ ]:
my_prompt = "파이썬 리스트와 튜플의 차이를 초보자에게 설명해줘. 예시 코드도 하나 보여줘."

print(ask(my_prompt))

> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [ ]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [ ]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

In [ ]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정:"""

print("[Few-shot]")
print(ask(few_shot))

> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [ ]:
prompt = "AI를 주제로 짧은 삼행시를 지어줘."

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

In [ ]:
# max_output_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_output_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_output_tokens=30))

## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [ ]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    prompt = "\n".join([f"{item['role']}: {item['content']}" for item in history])
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(max_output_tokens=1024),
    )
    answer = response.text
    history.append({"role": "assistant", "content": answer})
    return answer

print("✅ chat() 함수 준비 완료")

In [ ]:
print(chat("내 이름은 코알라야. 기억해줘."))

In [ ]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [ ]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [ ]:
# 과제 1 예시 (여기서부터 직접 수정해 보세요)
guide_system = "너는 밝고 친근한 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다."

print(ask("부산 여행 코스를 추천해줘.", system=guide_system))

In [ ]:
# 과제 2 · Few-shot 한영 단어 번역기
# 예시를 먼저 보여준 뒤, 마지막 한글 단어를 영어 단어로만 답하게 합니다.

translation_prompt = """다음 예시처럼 한글 단어를 영어 단어 하나로만 번역해줘.
문장이나 설명은 쓰지 말고 영어 단어만 출력해.

한글: 사과
영어: apple

한글: 학교
영어: school

한글: 컴퓨터
영어: computer

한글: 바다
영어:"""

print("[과제 2 결과]")
print(ask(translation_prompt, temperature=0.0, max_output_tokens=20))


# 과제 3 · 멀티턴 대화 예시
# 이전 대화 내용을 history에 쌓아 두고, 뒤 질문에서 기억하는지 확인합니다.
history.clear()
print("\n[과제 3 결과]")
print(chat("내가 좋아하는 여행지는 부산이야. 기억해줘."))
print(chat("내가 좋아하는 여행지는 어디라고 했지?"))


---

### 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `genai.configure(api_key=...)` 로 Gemini API 키를 설정한다
2. `model.generate_content(...)` 로 LLM에 요청을 보낸다
3. `system` 역할 지시를 포함해 답변 스타일을 조절한다
4. `temperature`·`max_output_tokens` 로 답변의 창의성과 길이를 조절한다
5. 대화를 기억시키려면 이전 기록을 함께 전달한다

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 Google AI Studio에서 삭제·재발급하세요.
